# Getting familiar with the synthetic biography data

This notebook follows the data from source lists to model input, then explains the experiment populations **A**, **B**, **C**, and **B′**. It is a read-and-inspect guide; it does not train a model or alter data files.

Run from the repository root (or let the first cell locate it). The repository environment should have the project dependencies installed (`uv sync`). The preprocessed artifact is `data/biography/preprocessed.npz`; if it is absent, run `uv run python -m src.data.preprocess_biography` first.

**Naming note:** A/B/C/B′ in the paper describe fact/value regions and training conditions. The implementation also has a held-out population literally named `dataC`. That is a different use of “C”; the notebook calls it **held-out dataC** where needed.

## 1. Locate the repository and load the preprocessed artifact

The `.npz` file contains tokenized source lists, templates, lengths, and vocabulary metadata. It does **not** contain a fixed table of complete biographies: those are assembled on demand by `BiographyDataset`. 

In [2]:
from pathlib import Path
import numpy as np

# Find the repository root from the current notebook directory.
_here = Path.cwd().resolve()
ROOT = next((p for p in [_here, *_here.parents] if (p / 'src/data/biography.py').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Could not find src/data/biography.py; start Jupyter inside this repository.')
DATA_PATH = ROOT / 'data/biography/preprocessed.npz'
if not DATA_PATH.exists():
    raise FileNotFoundError(f'{DATA_PATH} is missing. Run: uv run python -m src.data.preprocess_biography')

# Ensure imports work even if Jupyter was launched from a subdirectory.
import sys
sys.path.insert(0, str(ROOT))
raw = np.load(DATA_PATH, allow_pickle=True)
print('Repository:', ROOT)
print('Artifact:', DATA_PATH, f'({DATA_PATH.stat().st_size:,} bytes)')
print('Arrays stored:', len(raw.files))

Repository: /home/ogomi/PhD/spurious-forgetting-calibration-check
Artifact: /home/ogomi/PhD/spurious-forgetting-calibration-check/data/biography/preprocessed.npz (100,128 bytes)
Arrays stored: 65


## 2. What exists before preprocessing?

The source file `src/data/biography_corpus.py` defines six value pools and 25 English plus 25 French sentence templates for each attribute. `[X]` marks a person’s name and `[Y]` the attribute value. Name pools provide first, middle, and last names. The raw values are curated synthetic strings, not retrieved biographies or claims about real people.

In [ ]:
from src.data.biography_corpus import (
    ATTRIBUTE_NAMES, TEMPLATES, TEMPLATES_FR, BIRTH_YEARS, CITIES,
    UNIVERSITIES, FIELDS, COMPANIES, FIRST_NAMES, MIDDLE_NAMES, LAST_NAMES,
)

value_pools = {
    'birthdate': BIRTH_YEARS, 'birthplace': CITIES,
    'university': UNIVERSITIES, 'major': FIELDS,
    'company': COMPANIES, 'work_location': CITIES,
}
print('Attribute | raw values | English templates | French templates')
for attr in ATTRIBUTE_NAMES:
    print(f'{attr:14} | {len(value_pools[attr]):10} | {len(TEMPLATES[attr]):17} | {len(TEMPLATES_FR[attr]):16}')
print(f'Name pools: first={len(FIRST_NAMES)}, middle={len(MIDDLE_NAMES)}, last={len(LAST_NAMES)}')
print('Example raw values:')
for attr in ATTRIBUTE_NAMES:
    print(f'  {attr}: {value_pools[attr][:5]}')
print('Example template:', TEMPLATES['birthplace'][0])
print('French counterpart:', TEMPLATES_FR['birthplace'][0])

## 3. Preprocessing: strings become a compact token vocabulary

`src/data/preprocess_biography.py` loads GPT-2 BPE, adds `[PAD]`, `[BOS]`, `[X]`, `[Y]`, and makes each birth year one token. It tokenizes values, names, and templates. For each attribute pool, it keeps the first value encountered for each distinct **first BPE token**; later values sharing that initial token are removed. This makes the first value token sufficient to identify the attribute’s selected value. The filter is per pool, not across all six attributes.

Finally it prunes GPT-2’s large vocabulary to tokens actually used in the corpus and remaps token IDs. Template tokenization preserves `[X]` / `[Y]` as single placeholder IDs so the generator can insert names and values later.

In [ ]:
def scalar(key):
    x = raw[key]
    return x.item() if getattr(x, 'shape', None) == () else x

print('Preprocessed metadata:')
for key in ['vocab_size', 'english_vocab_size', 'max_sentence_len', 'max_sentence_len_fr', 'max_name_len', 'pad_id', 'bos_id', 'x_id', 'y_id']:
    if key in raw.files:
        print(f'  {key}: {scalar(key)}')
print('Attribute value counts after first-token filter:')
for k, attr in enumerate(ATTRIBUTE_NAMES):
    print(f'  {attr:14}: raw {len(value_pools[attr]):3} -> kept {len(raw[f"attr_{k}_value_strings"]):3}')
print('Vocabulary pruning: GPT-2 base vocabulary ->', int(raw['vocab_size']), 'used tokens (including special tokens)')

### Look at actual kept values and tokenizations

The saved token IDs are remapped IDs. `token_strings[id]` gives the corresponding token text. Some BPE token strings show whitespace markers; that is tokenizer notation, not necessarily a literal character in a printed biography.

In [ ]:
token_strings = raw['token_strings'].tolist()
def token_pieces(ids, length):
    return [token_strings[int(i)] for i in ids[:int(length)]]

for k, attr in enumerate(ATTRIBUTE_NAMES):
    strings = raw[f'attr_{k}_value_strings'].tolist()
    ids = raw[f'attr_{k}_value_token_ids']
    lens = raw[f'attr_{k}_value_token_lengths']
    print(f'\n{attr}:')
    for i in range(min(4, len(strings))):
        print(f'  {strings[i]!r:24} -> {token_pieces(ids[i], lens[i])}')

## 4. What is one generated biography?

A population assigns each synthetic person a unique first/middle/last name combination, a value support for each of the six attributes, and a randomized split of that attribute’s 25 templates into training and evaluation templates (defaults: 20 train, 5 eval). With the default `support_size=1`, a person has one fixed value per attribute. A generated sequence starts with `[BOS]` and the name, then includes all six attribute sentences in a random order. The exact template and sentence order vary across samples.

The next cell creates a **tiny illustration** with four people: two assigned to value half X and two to Y. It follows the same builder/generator code, but the sample names/values are a seeded demonstration, not the full experiment’s fixed 2,000/2,000/500/500 groups.

In [ ]:
from src.data.biography import BiographyPopulation, BiographyDataset
from src.experiments.knowledge_injection import build_partition, assign_values

# Miniature deterministic example; full experiment defaults are documented below.
pop_demo = BiographyPopulation(str(DATA_PATH), num_people=4, seed=42, support_size=1, num_train_templates=20)
halves_demo = build_partition(pop_demo, seed=7)
rng_demo = np.random.default_rng(942)
assign_values(pop_demo, np.array([0, 1]), halves_demo, 'X', rng_demo, support_size=1)
assign_values(pop_demo, np.array([2, 3]), halves_demo, 'Y', rng_demo, support_size=1)
ds_demo = BiographyDataset(pop_demo, np.array([0]), name='one-person-demo', seed=44)
seqs, masks, person_ids, langs = ds_demo._generate(1, eval_templates=False, rng=np.random.default_rng(45))

def decode(ids):
    # Convert remapped tokens back to readable text, skipping padding.
    return ''.join(token_strings[int(i)] for i in ids if int(i) != pop_demo.pad_id).replace('Ġ', ' ')

print('Person ID:', int(person_ids[0]))
print('Readable sequence (token strings concatenated; spacing is approximate):')
print(decode(seqs[0]))
print('Sequence array shape:', seqs.shape, '| configured seq_len:', pop_demo.seq_len, '| prompt_len:', pop_demo.prompt_len)
print('Six sentences are generated; any remaining positions are PAD.')

## 5. The final training example: shifted tokens and role masks

The model receives `inputs = sequence[:-1]` and predicts `targets = sequence[1:]` (next-token prediction). `eval_mask` / the batch mask identifies which target positions matter for which metric. The role labels are defined in `src/data/biography.py`: `0` prompt padding, `7` names and template text, `+1…+6` the first token of each attribute value, `-1…-6` continuation tokens, and `8` trailing padding/end-of-document. The special first-token labels are why the filter above matters. Reported first-token accuracy evaluates the attribute’s first generated value token.

The next cell displays a compact alignment for the generated example, trimming trailing padding. For arrays, positions in the mask align to the shifted target sequence.

In [ ]:
# Build a readable table for the first generated row, through its last non-padding token.
seq = seqs[0]
mask = masks[0]
end = int(np.flatnonzero(seq != pop_demo.pad_id)[-1]) + 1 if np.any(seq != pop_demo.pad_id) else 0
role_names = {0: 'prompt pad', 7: 'name/template', 8: 'end padding'}
for k, attr in enumerate(ATTRIBUTE_NAMES, start=1):
    role_names[k] = f'{attr}: first value token'
    role_names[-k] = f'{attr}: value continuation'
print('pos | token | role')
for pos in range(min(end, 100)):
    tok = token_strings[int(seq[pos])]
    print(f'{pos:3} | {tok!r:20} | {role_names.get(int(mask[pos]), str(mask[pos]))}')
if end > 100: print(f'... {end-100} more non-padding positions')

## 6. A, B, C, and B′: what the experiment means

The transformer experiment partitions each attribute’s filtered value pool into two disjoint halves, X and Y, using a seeded random permutation. People are separate from values: A and B are different synthetic people, while their facts are made related or unrelated by restricting which value half they can use. With the repository defaults, there are 2,000 A people, 2,000 ballast people (called `ballast` or D in some code/comments), 500 B people, and 500 held-out C people.

| Name | Role in the experiment | Values used | Seen in training? |
|---|---|---|---|
| **A** (`dataA`) | Old facts whose recall is tracked | X | Pretraining |
| **Ballast / D** (`ballast`) | Control facts occupying the other value region | Y | Pretraining |
| **B** (`dataB`) | New facts used for finetuning/injection | Depends on condition; B′ uses Y | Pretraining no; injection yes |
| **C condition** | Comparison condition for new facts | B draws from the whole X∪Y pool (`all_values`) | Injection yes |
| **Held-out dataC** | Separate people, evaluated for generalization/confidence analysis | Default X (`c_half='X'`) | Neither phase |
| **B′** | Paper’s disjoint new-fact condition: shares value region with ballast/B, not A | Y | Injection yes |

The README’s “C” denotes the **all-values comparison condition**, while `dataC` in the implementation is an optional, distinct held-out group. Both concepts exist. In code, `B′` is still `dataB`; the selected injection condition is `disjoint`, which maps B to Y. The default `high_overlap` maps B to X, `partial` mixes X/Y by person and attribute, and `all_values` samples across both halves. A and ballast are pretrained together; B is injected later without replay. Held-out dataC is never in the pretraining or injection training sampler.

In [ ]:
from src.experiments.knowledge_injection import KIConfig, B_HALF

cfg = KIConfig()
print('Default experiment population sizes:')
for name in ['num_a', 'num_ballast', 'num_b', 'num_c']:
    print(f'  {name}: {getattr(cfg, name):,}')
print('Total synthetic people:', cfg.num_a + cfg.num_ballast + cfg.num_b + cfg.num_c)
print('Condition -> B value assignment:', B_HALF)
print('Default held-out C half:', cfg.c_half)
print('Default attribute support size:', cfg.data.support_size)
print('Train/eval templates per attribute:', cfg.data.num_train_templates, '/', 25-cfg.data.num_train_templates)

### Check the partition sizes

For the usual 50/50 split, X and Y contain approximately half the retained values each (one half can be one item larger when the count is odd). This cell reports the exact split size using the same partition function and seed as the main experiment’s default configuration.

In [ ]:
from src.experiments.knowledge_injection import build_partition

# Use a tiny population only to initialize the source value counts; the partition itself
# depends on per-attribute pool sizes and the configured seed.
pop_stats = BiographyPopulation(str(DATA_PATH), num_people=1, seed=42)
halves = build_partition(pop_stats, seed=cfg.partition_seed)
print('attribute | retained | X | Y')
for k, attr in enumerate(ATTRIBUTE_NAMES):
    print(f'{attr:14} | {len(pop_stats.attr_value_ids[k]):8} | {len(halves[k][0]):3} | {len(halves[k][1]):3}')

## 7. A few useful statistics and interpretation

The next summaries are corpus-level facts, not model performance. Sentence lengths are variable; the stored maximum values are used to allocate fixed-size model sequences. Names are repeated inside each attribute sentence, and all six attributes appear once per generated biography. This is a controlled, compositional task: person identity is conveyed by the name prompt; the model must generate synthetic attribute values in multiple phrasings. It is not a natural biography benchmark, and the “facts” are randomly assigned rather than factual.

The first-token filter also means the value classes are deliberately easy to distinguish at the first answer token. That supports attribute-level recall metrics, but it makes this setup different from unconstrained language generation.

In [ ]:
print('Corpus / tokenization summary')
print(f'  attributes: {len(ATTRIBUTE_NAMES)}')
print(f'  templates per attribute: {len(TEMPLATES[ATTRIBUTE_NAMES[0]])} English + {len(TEMPLATES_FR[ATTRIBUTE_NAMES[0]])} French')
print(f'  retained attribute values total: {sum(len(raw[f"attr_{k}_value_strings"]) for k in range(6)):,}')
print(f'  used tokens: {int(raw["vocab_size"]):,} (English prefix: {int(raw["english_vocab_size"]):,})')
print(f'  max name tokens (three name parts concatenated): {int(raw["max_name_len"])}')
print(f'  max sentence tokens: English {int(raw["max_sentence_len"])}, French {int(raw["max_sentence_len_fr"])}')
print(f'  prompt tokens including BOS: {1 + int(raw["max_name_len"])}')
print('  one biography contains six attribute sentences, padded to a fixed sequence length')

## 8. Where to look next

- `src/data/biography_corpus.py`: raw template and value lists.
- `src/data/preprocess_biography.py`: tokenizer setup, first-token filtering, vocabulary pruning, and `.npz` writing.
- `src/data/biography.py`: population assignment, sequence generation, evaluation templates, shifting, and masks.
- `src/experiments/knowledge_injection.py`: X/Y partitioning and A/ballast/B/held-out-C construction.
- `src/experiments/mlpfree_pretrain.py` and `src/experiments/mlpfree_injection.py`: concrete pretraining and B-injection recipes.
- `README.md`: short paper-level overview and standard commands.

The preprocessed `.npz` is a compact vocabulary-and-source-data artifact. The experiment’s A/B/etc. groups are generated from a seed/configuration at runtime; they are not four prebuilt CSVs. For reproducibility, keep the seeds, value partition, preprocessing artifact, and experiment configuration together.